# Course Overview: MIDAS Regression and Nowcasting

**Estimated time:** 2 minutes

This notebook provides a visual tour of the four core concepts in this course. Run it top-to-bottom to see what you will build.

| Module | Topic | Core Skill |
|--------|-------|------------|
| 00 | Mixed-frequency data | Build MIDAS matrices |
| 01 | MIDAS fundamentals | Beta polynomial weights |
| 02 | Estimation & inference | Profile NLS, HAC SEs |
| 03 | Nowcasting | Ragged edge, RMSE by vintage |
| 04 | Dynamic Factor Models | PCA factors, FA-MIDAS |

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

# ============================================================
# Visual 1: The Mixed-Frequency Problem
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Simulated quarterly GDP and monthly IP
np.random.seed(0)
T_q = 40; T_m = T_q * 3
ip_monthly   = np.cumsum(0.2 * np.random.randn(T_m))
gdp_quarterly = np.array([ip_monthly[i*3:(i+1)*3].mean() for i in range(T_q)]) + \
                0.3 * np.random.randn(T_q)

ax = axes[0, 0]
ax.plot(range(T_m), ip_monthly, color='steelblue', linewidth=1, alpha=0.7, label='Monthly IP')
q_times = [i*3+1.5 for i in range(T_q)]
ax.scatter(q_times, gdp_quarterly, color='tomato', s=60, zorder=5, label='Quarterly GDP')
for i in range(0, T_m, 3):
    ax.axvline(i, color='gray', linewidth=0.4, alpha=0.4)
ax.set_title('Mixed-Frequency Data: IP (monthly) + GDP (quarterly)')
ax.set_xlabel('Month')
ax.legend(fontsize=9)

# ============================================================
# Visual 2: Beta Polynomial Weight Shapes
# ============================================================
ax2 = axes[0, 1]
K = 12; j = np.arange(K); x = (j + 0.5) / K
specs = [
    (1.0, 1.0, 'Equal weights (θ₁=1, θ₂=1)', 'gray'),
    (1.2, 4.0, 'Mild front-load (θ₁=1.2, θ₂=4)', 'steelblue'),
    (1.5, 7.0, 'Strong front-load (θ₁=1.5, θ₂=7)', 'tomato'),
    (2.0, 2.0, 'Hump-shaped (θ₁=2, θ₂=2)', 'green'),
]
for t1, t2, label, color in specs:
    raw = beta_dist.pdf(1 - x, t1, t2)
    w = raw / raw.sum()
    ax2.plot(j, w, linewidth=2.5, color=color, label=label)
ax2.axhline(1/K, color='gray', linestyle=':', linewidth=1)
ax2.set_title('Beta Polynomial Weight Shapes')
ax2.set_xlabel('Lag j (j=0 = most recent month)')
ax2.set_ylabel('Weight')
ax2.legend(fontsize=7.5)

# ============================================================
# Visual 3: Profile SSE Contour
# ============================================================
ax3 = axes[1, 0]
# Simulate a simple profile SSE landscape
T_sim = 80; K_sim = 9
np.random.seed(42)
x_m = np.random.randn(T_sim, K_sim)
w_true = beta_dist.pdf(1 - (np.arange(K_sim)+0.5)/K_sim, 1.4, 4.5)
w_true /= w_true.sum()
Y_sim = 0.4 + 0.6 * (x_m @ w_true) + 0.5 * np.random.randn(T_sim)

def psse_sim(t1, t2):
    if t1 <= 0.01 or t2 <= 0.01: return np.nan
    x = (np.arange(K_sim)+0.5)/K_sim
    raw = beta_dist.pdf(1-x, t1, t2)
    s = raw.sum()
    w = raw/s if s > 1e-12 else np.ones(K_sim)/K_sim
    xw = x_m @ w; xc = xw - xw.mean(); yc = Y_sim - Y_sim.mean()
    ss = xc@xc
    if ss < 1e-12: return np.sum((Y_sim-Y_sim.mean())**2)
    b = yc@xc/ss; a = Y_sim.mean() - b*xw.mean()
    return np.sum((Y_sim-a-b*xw)**2)

t1_grid = np.linspace(0.3, 3.5, 25)
t2_grid = np.linspace(0.3, 9.0, 30)
sse_grid = np.array([[psse_sim(t1, t2) for t2 in t2_grid] for t1 in t1_grid])
sse_grid = np.where(np.isnan(sse_grid), sse_grid.max(), sse_grid)

cs = ax3.contourf(t2_grid, t1_grid, sse_grid, levels=20, cmap='RdYlGn_r')
plt.colorbar(cs, ax=ax3, label='SSE')
i_opt = np.unravel_index(np.nanargmin(sse_grid), sse_grid.shape)
ax3.scatter([t2_grid[i_opt[1]]], [t1_grid[i_opt[0]]],
            marker='*', s=200, color='white', zorder=5, label='Minimum')
ax3.set_xlabel('θ₂'); ax3.set_ylabel('θ₁')
ax3.set_title('Profile SSE Landscape — Finding θ*')
ax3.legend(fontsize=9)

# ============================================================
# Visual 4: Nowcast Vintages
# ============================================================
ax4 = axes[1, 1]
# Illustrative nowcast evolution
quarters = np.arange(10)
actuals  = np.array([0.5, 0.8, 0.3, 1.2, 0.7, -0.5, 0.9, 1.1, 0.4, 0.6])
h2_ncast = actuals + 0.3 * np.random.randn(10) + 0.15  # 1-month (worst)
h1_ncast = actuals + 0.2 * np.random.randn(10) + 0.08  # 2-month
h0_ncast = actuals + 0.12 * np.random.randn(10)         # 3-month (best)

ax4.plot(quarters, actuals, 'ko-', linewidth=2, markersize=7, label='Actual GDP', zorder=5)
ax4.plot(quarters, h2_ncast, color='#2ca02c', linewidth=1.5, linestyle=':', marker='s',
         markersize=5, label='1-month nowcast', alpha=0.8)
ax4.plot(quarters, h1_ncast, color='#ff7f0e', linewidth=1.5, linestyle='--', marker='^',
         markersize=5, label='2-month nowcast', alpha=0.8)
ax4.plot(quarters, h0_ncast, color='#1f77b4', linewidth=2, linestyle='-', marker='o',
         markersize=6, label='3-month nowcast')
ax4.axhline(0, color='gray', linewidth=0.7, alpha=0.5)
ax4.set_xlabel('Quarter'); ax4.set_ylabel('GDP growth (%)')
ax4.set_title('Nowcast Evolution: 1→2→3 Month Vintage')
ax4.legend(fontsize=8)

plt.suptitle('Course Overview: MIDAS Regression and Nowcasting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('course_overview.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nCourse overview complete.")
print("Start with Module 00 → Module 01 → Module 02 → Module 03 → Module 04")

In [ ]:
apply_course_theme()
apply_plot_theme()